# Week 6 Example Solution: Build a Defensible Classifier

**Task:** Classify participants as having elevated depression (PHQ-9 ≥ 5) vs minimal symptoms using the Boston College COVID-19 Sleep & Well-Being dataset.

**Models:** Baseline → Logistic Regression → Decision Tree → Random Forest

**Key results:** Random Forest AUC = 0.920, top predictors are PANAS negative affect and GAD-7 anxiety.

---

## Part 1: Data Loading and Preparation

We have three CSV files to load, process, and merge into a single dataset.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

print('All imports loaded successfully!')

In [ ]:
# Cell 2: Load the three data files
data_dir = '../data/boston_college'

daily = pd.read_csv(f'{data_dir}/daily_survey.csv')
demo = pd.read_csv(f'{data_dir}/demographics.csv')
r1 = pd.read_csv(f'{data_dir}/round1_assessment.csv')

print(f'Daily survey: {daily.shape[0]:,} entries from {daily.sub_id.nunique():,} participants')
print(f'Demographics: {demo.shape[0]:,} participants')
print(f'Round 1:      {r1.shape[0]:,} entries from {r1.sub_id.nunique():,} unique participants')

In [ ]:
# Cell 3: Aggregate daily survey per participant
# Each participant has multiple daily entries — average them
agg_cols = ['PHQ9', 'PANAS_PA', 'PANAS_NA', 'TST', 'SE',
            'sleepdiary_sleeplatency', 'sleepdiary_wakes',
            'exercise', 'alcohol_bev', 'isolation', 'stress',
            'worry_scale', 'panas_sad3', 'panas_happy3', 'people_contact']

daily_with_phq = daily[daily['PHQ9'].notna()]
daily_agg = daily_with_phq.groupby('sub_id')[agg_cols].mean()

print(f'After aggregation: {daily_agg.shape[0]:,} participants with PHQ-9 data')

In [ ]:
# Cell 4: Score Big Five personality traits from BFI-2 items
r1_dedup = r1.drop_duplicates(subset='sub_id', keep='first').copy()
print(f'Round 1 after dedup: {len(r1_dedup)} (removed {len(r1) - len(r1_dedup)} duplicates)')

def score_trait(df, pos_items, neg_items):
    """Score a Big Five trait from positive and reverse-scored items."""
    pos = df[[f'big5_{i}' for i in pos_items]].values
    neg = 6 - df[[f'big5_{i}' for i in neg_items]].values
    all_items = np.concatenate([pos, neg], axis=1)
    return np.nanmean(all_items, axis=1)

r1_dedup['Extraversion'] = score_trait(r1_dedup, [1, 11, 21], [6, 16, 26])
r1_dedup['Agreeableness'] = score_trait(r1_dedup, [7, 17, 27], [2, 12, 22])
r1_dedup['Conscientiousness'] = score_trait(r1_dedup, [3, 13, 23], [8, 18, 28])
r1_dedup['Neuroticism'] = score_trait(r1_dedup, [4, 14, 24], [9, 19, 29])
r1_dedup['Openness'] = score_trait(r1_dedup, [5, 15, 25], [10, 20, 30])

# GAD-7 total
gad_cols = [f'gad_{i}' for i in range(1, 8)]
r1_dedup['GAD7_total'] = r1_dedup[gad_cols].sum(axis=1)

print('Big Five traits and GAD-7 scored successfully!')

In [ ]:
# Cell 5: Merge all three dataframes
merged = daily_agg.merge(
    demo[['sub_id', 'age1', 'bio_sex']],
    left_index=True, right_on='sub_id', how='inner'
)
print(f'After merge with demographics: {merged.shape}')

merged = merged.merge(
    r1_dedup[['sub_id', 'Extraversion', 'Agreeableness', 'Conscientiousness',
              'Neuroticism', 'Openness', 'GAD7_total']],
    on='sub_id', how='inner'
)
merged = merged.drop_duplicates(subset='sub_id')
print(f'After merge with Round 1: {merged.shape}')

In [ ]:
# Cell 6: Create binary target
merged['depression_elevated'] = (merged['PHQ9'] >= 5).astype(int)

print('Class balance:')
print(merged['depression_elevated'].value_counts())
print()
print('Proportions:')
print(merged['depression_elevated'].value_counts(normalize=True).round(3))

In [ ]:
# Cell 7: Define features and target
feature_cols = ['PANAS_PA', 'PANAS_NA', 'TST', 'SE',
                'sleepdiary_sleeplatency', 'sleepdiary_wakes',
                'exercise', 'alcohol_bev', 'isolation', 'stress',
                'worry_scale', 'panas_sad3', 'panas_happy3',
                'age1', 'bio_sex',
                'Extraversion', 'Agreeableness', 'Conscientiousness',
                'Neuroticism', 'Openness', 'GAD7_total']

X = merged[feature_cols].copy()
y = merged['depression_elevated'].copy()

# Handle missing values
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols, index=X.index)

print(f'Features: {len(feature_cols)} columns')
print(f'Samples: {len(X)}')
print(f'Target: {y.value_counts().to_dict()}')

## Part 2: Classification Pipeline

Now we build and compare four classifiers, from the majority-class baseline through to Random Forest.

In [ ]:
# Cell 8: Train/test split (stratified to keep class proportions equal)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}')
print(f'Test class balance:  {y_test.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Cell 9: Reusable evaluation function (the refactoring win!)
#
# Instead of repeating the same metric-computation code for each model,
# we created this function. Originally we had ~40 lines of duplicated code.

def evaluate_model(model, X_tr, X_te, y_tr, y_te, name="Model"):
    """Fit a model and return a dictionary of classification metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    metrics = {
        'name': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'f1': f1_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall': recall_score(y_te, y_pred),
    }
    
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics['auc'] = roc_auc_score(y_te, y_prob)
    else:
        metrics['auc'] = None
    
    return metrics

print('evaluate_model() function defined.')

In [ ]:
# Cell 10: Baseline — always predict the majority class
baseline = DummyClassifier(strategy='most_frequent')
baseline_metrics = evaluate_model(baseline, X_train, X_test, y_train, y_test, 'Baseline')

print(f"Baseline (majority class):")
print(f"  Accuracy: {baseline_metrics['accuracy']:.3f}")
print(f"  (Always predicts 'elevated' — this is the bar to beat)")

In [ ]:
# Cell 11: Logistic Regression (uses scaled features)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr_metrics = evaluate_model(lr, X_train_scaled, X_test_scaled, y_train, y_test, 'Logistic Regression')

print(f"Logistic Regression:")
print(f"  Accuracy: {lr_metrics['accuracy']:.3f}")
print(f"  F1:       {lr_metrics['f1']:.3f}")
print(f"  AUC:      {lr_metrics['auc']:.3f}")
print(f"  Precision: {lr_metrics['precision']:.3f}")
print(f"  Recall:    {lr_metrics['recall']:.3f}")

In [ ]:
# Cell 12: Decision Tree (max_depth=5 to prevent overfitting)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_metrics = evaluate_model(dt, X_train, X_test, y_train, y_test, 'Decision Tree')

print(f"Decision Tree (depth=5):")
print(f"  Accuracy: {dt_metrics['accuracy']:.3f}")
print(f"  F1:       {dt_metrics['f1']:.3f}")
print(f"  AUC:      {dt_metrics['auc']:.3f}")

In [ ]:
# Cell 13: Random Forest (100 trees, max_depth=10)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_metrics = evaluate_model(rf, X_train, X_test, y_train, y_test, 'Random Forest')

print(f"Random Forest (100 trees, depth=10):")
print(f"  Accuracy: {rf_metrics['accuracy']:.3f}")
print(f"  F1:       {rf_metrics['f1']:.3f}")
print(f"  AUC:      {rf_metrics['auc']:.3f}")

In [ ]:
# Cell 14: Cross-validation to check stability
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-fold cross-validation (accuracy):')
for name, model, data in [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42), 
     scaler.fit_transform(X)),
    ('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42), X),
    ('Random Forest', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42), X),
]:
    scores = cross_val_score(model, data, y, cv=cv, scoring='accuracy')
    print(f'  {name}: {scores.mean():.3f} +/- {scores.std():.3f}')

## Part 3: Model Comparison and Analysis

In [ ]:
# Cell 15: Model comparison chart
all_metrics = [baseline_metrics, lr_metrics, dt_metrics, rf_metrics]
colours = ['#8C8C8C', '#4A90D9', '#5BA55B', '#E8873D']
model_names = [m['name'] for m in all_metrics]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
axes[0].bar(model_names, [m['accuracy'] for m in all_metrics],
            color=colours, edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylim(0.4, 0.9)
for i, m in enumerate(all_metrics):
    axes[0].text(i, m['accuracy'] + 0.01, f"{m['accuracy']:.3f}",
                 ha='center', fontsize=10, fontweight='bold')

# F1
f1_vals = [m['f1'] if m['name'] != 'Baseline' else 0 for m in all_metrics]
axes[1].bar(model_names, f1_vals, color=colours, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, 0.95)
for i, (m, v) in enumerate(zip(all_metrics, f1_vals)):
    if v > 0:
        axes[1].text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=10, fontweight='bold')

# AUC
auc_vals = [m['auc'] if m['auc'] is not None else 0 for m in all_metrics]
axes[2].bar(model_names, auc_vals, color=colours, edgecolor='white', linewidth=0.5)
axes[2].set_ylabel('AUC', fontsize=12)
axes[2].set_title('Area Under ROC Curve', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=15)
axes[2].set_ylim(0, 1.0)
for i, (m, v) in enumerate(zip(all_metrics, auc_vals)):
    if v > 0:
        axes[2].text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Feature importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bar_colours = ['#4A90D9' if v >= importances.quantile(0.75) else '#8C8C8C'
               for v in importances_sorted]
importances_sorted.plot(kind='barh', ax=ax, color=bar_colours, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Random Forest Feature Importance: What Predicts Depression?',
             fontsize=14, fontweight='bold')

for i, (val, name) in enumerate(zip(importances_sorted, importances_sorted.index)):
    ax.text(val + 0.002, i, f"{val:.3f}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nTop 10 features:')
for feat, imp in importances.sort_values(ascending=False).head(10).items():
    print(f'  {feat:30s} {imp:.3f}')

In [ ]:
# Cell 17: ROC curves for all three real classifiers
fig, ax = plt.subplots(figsize=(8, 8))

# Get probabilities from each model
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
dt_probs = dt.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]

for probs, name, colour, auc_val in [
    (lr_probs, 'Logistic Regression', '#4A90D9', lr_metrics['auc']),
    (dt_probs, 'Decision Tree', '#5BA55B', dt_metrics['auc']),
    (rf_probs, 'Random Forest', '#E8873D', rf_metrics['auc']),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, color=colour, linewidth=2, label=f'{name} (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: All Classifiers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

## Part 4: Threshold Analysis

The default threshold is 0.5, but for a depression screener we might want a lower threshold to catch more cases (higher recall), even at the cost of more false positives (lower precision).

In [ ]:
# Cell 18: Threshold analysis — how do metrics change with different thresholds?
thresholds = np.arange(0.1, 0.91, 0.01)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (rf_probs >= t).astype(int)
    if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
        precisions.append(np.nan)
        recalls.append(np.nan)
        f1s.append(np.nan)
    else:
        precisions.append(precision_score(y_test, y_pred_t))
        recalls.append(recall_score(y_test, y_pred_t))
        f1s.append(f1_score(y_test, y_pred_t))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, precisions, color='#4A90D9', linewidth=2, label='Precision')
ax.plot(thresholds, recalls, color='#A71930', linewidth=2, label='Recall')
ax.plot(thresholds, f1s, color='#5BA55B', linewidth=2, label='F1 Score')
ax.axvline(x=0.5, color='grey', linewidth=1, linestyle='--', alpha=0.7, label='Default (0.5)')
ax.axvline(x=0.3, color='#E8873D', linewidth=1, linestyle='--', alpha=0.7, label='Screening (0.3)')

ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Threshold Analysis: Random Forest', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='center left')
ax.set_xlim(0.1, 0.9)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 19: Compare specific thresholds
print('Threshold comparison (Random Forest):')
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 42)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (rf_probs >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print(f'{t:>10.1f} {p:>10.3f} {r:>10.3f} {f:>10.3f}')

In [ ]:
# Cell 20: Confusion matrices at two thresholds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, threshold, title in zip(axes, [0.3, 0.5],
                                 ['Threshold = 0.3 (Screening)', 'Threshold = 0.5 (Default)']):
    y_pred_t = (rf_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Minimal', 'Elevated'],
                yticklabels=['Minimal', 'Elevated'],
                annot_kws={'size': 16, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    ax.text(0.5, -0.15, f'Precision={p:.2f}  Recall={r:.2f}  F1={f:.2f}',
            transform=ax.transAxes, ha='center', fontsize=11, color='#333')

plt.tight_layout()
plt.show()

## Part 5: Interpretation and Ethics

In [ ]:
# Cell 21: Full classification report
print('Full Classification Report (Random Forest, threshold=0.5)')
print('=' * 55)
y_pred_final = rf.predict(X_test)
print(classification_report(y_test, y_pred_final, target_names=['Minimal', 'Elevated']))

## Ethical Consideration

This model was trained on data collected during the early COVID-19 pandemic (March–June 2020) from a predominantly female, US-based sample. Depression rates during this period were likely elevated compared to normal times, meaning the model's learned patterns may not generalise to other contexts or populations.

If this model were deployed as a screening tool, it should only be used to **flag** individuals for follow-up support — never to **deny** access to mental health services. A false negative (telling someone they're fine when they're not) could delay treatment with real consequences. The model should supplement, not replace, clinical judgement.

In [ ]:
# Cell 22: Save figures (uncomment when ready)
# fig.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
# print('Figure saved!')